# News Category Classification

**Classification Project 37 of 100** — Classify news articles into categories using tabularized text features.

EDA -> TF-IDF features -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.


## 2. Project Overview

Classify news articles/headlines into topic categories using tabularized text features (TF-IDF). Multi-class classification on text-derived features.

Multi-class classification with 8+ categories.


## 3. Learning Objectives

1. Multi-class text classification
2. TF-IDF on news headlines/descriptions
3. Category reduction strategies
4. Multi-class evaluation metrics (macro F1)
5. LazyPredict and FLAML for multi-class
6. News domain context


## 4. Problem Statement

> Given a news headline/description, predict its category.

Multi-class classification. F1-macro is primary.


## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| News platforms | Auto-categorization |
| Readers | Content discovery |
| ML learners | Multi-class text classification |
| Content teams | Workflow automation |


## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | rmisra/news-category-dataset |
| Rows | ~200k |
| Features | Headline, short description |
| Target | Category |
| Classes | 8+ (top categories) |


## 7. Dataset Source and License Notes

- Kaggle: rmisra/news-category-dataset
- HuffPost news articles
- License: Check Kaggle page.


## 8. Environment Setup


In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")


## 9. Imports


In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, ConfusionMatrixDisplay, RocCurveDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")


## 10. Configuration / Constants


In [ ]:
KAGGLE_SLUG = 'rmisra/news-category-dataset'
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 120; MAX_ROWS = 40000


## 11. Dataset Download or Loading


In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f'Downloaded to: {path}')
except Exception as e:
    raise RuntimeError(f'Download failed: {e}')
import glob
files = glob.glob(os.path.join(path,'**','*'), recursive=True)
json_files = [f for f in files if f.endswith('.json')]
csv_files = [f for f in files if f.endswith('.csv')]
if json_files:
    df_raw = pd.read_json(sorted(json_files, key=os.path.getsize, reverse=True)[0], lines=True)
elif csv_files:
    df_raw = pd.read_csv(sorted(csv_files, key=os.path.getsize, reverse=True)[0])
else:
    raise FileNotFoundError(f'No data files in {path}')
print(f'Shape: {df_raw.shape}')
df_raw.head()


## 12. Data Validation Checks


In [ ]:
print(f"Missing: {df_raw.isnull().sum().sum()}")
df = df_raw.copy()
tgt = [c for c in df.columns if any(h in c.lower() for h in ['category','label','class','target'])]
TARGET = tgt[0] if tgt else df.columns[-1]
txt = [c for c in df.columns if df[c].dtype == 'O' and c != TARGET]
txt_long = [c for c in txt if df[c].str.len().mean() > 30]
if len(txt_long) >= 2:
    TEXT_COL = txt_long[0]
    DESC_COL = txt_long[1]
    df['FULL_TEXT'] = df[TEXT_COL].fillna('') + ' ' + df[DESC_COL].fillna('')
    TEXT_COL = 'FULL_TEXT'
elif txt_long:
    TEXT_COL = txt_long[0]
else:
    TEXT_COL = txt[0] if txt else df.columns[1]
print(f'Target: {TARGET}, Text: {TEXT_COL}')
keep = [TARGET, TEXT_COL]
df = df[keep].dropna().drop_duplicates().reset_index(drop=True)
# Keep top N categories if too many
if df[TARGET].nunique() > 10:
    top_cats = df[TARGET].value_counts().head(8).index
    df = df[df[TARGET].isin(top_cats)].reset_index(drop=True)
    print(f'Kept top 8 categories')
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED).reset_index(drop=True)
print(f'Shape: {df.shape}, Classes: {df[TARGET].nunique()}')
print(df[TARGET].value_counts())


## 13. Exploratory Data Analysis


In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=['steelblue','coral','seagreen','orange','purple'][:df[TARGET].nunique()])
ax.set_title(f'Target: {TARGET}'); plt.tight_layout(); plt.show()


In [ ]:
num_feats = df.select_dtypes(include=[np.number]).columns.tolist()
num_feats = [c for c in num_feats if c != TARGET]
if len(num_feats) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(12,8))
    for ax, col in zip(axes.flat, num_feats[:4]):
        df[col].dropna().hist(bins=30, ax=ax, alpha=0.7); ax.set_title(col)
    plt.tight_layout(); plt.show()


In [ ]:
cat_feats = df.select_dtypes(include=['object','category']).columns.tolist()
if cat_feats:
    for col in cat_feats[:4]: print(f"{col}: {df[col].nunique()} unique")
np_cols = [c for c in num_feats if c != TARGET]
if len(np_cols) >= 2:
    corr = df[np_cols].corr(numeric_only=True)
    fig, ax = plt.subplots(figsize=(10,8))
    sns.heatmap(corr.iloc[:min(12,len(corr)),:min(12,len(corr))], annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Heatmap'); plt.tight_layout(); plt.show()


## 14. Target Analysis

Multi-class with variable category sizes.


In [ ]:
print(df[TARGET].value_counts(normalize=True))


## 15. Train / Validation / Test Split


In [ ]:
X = df.drop(columns=[TARGET]); y = df[TARGET]
if y.dtype == object:
    le = LabelEncoder(); y = pd.Series(le.fit_transform(y), name=TARGET)
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')


## 16. Preprocessing Strategy

TF-IDF (500 features) + text length features for standard ML classifiers.


## 17. Feature Engineering


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
tfidf = TfidfVectorizer(max_features=500, stop_words='english', sublinear_tf=True)
X_train_txt = tfidf.fit_transform(X_train[TEXT_COL].fillna(''))
X_val_txt = tfidf.transform(X_val[TEXT_COL].fillna(''))
X_test_txt = tfidf.transform(X_test[TEXT_COL].fillna(''))
for d in [X_train, X_val, X_test]:
    d['TXT_LEN'] = d[TEXT_COL].fillna('').str.len()
    d['WORD_CT'] = d[TEXT_COL].fillna('').str.split().str.len()
hf = ['TXT_LEN','WORD_CT']
X_train = pd.DataFrame(sp.hstack([X_train_txt, sp.csr_matrix(X_train[hf].values)]).toarray())
X_val = pd.DataFrame(sp.hstack([X_val_txt, sp.csr_matrix(X_val[hf].values)]).toarray())
X_test = pd.DataFrame(sp.hstack([X_test_txt, sp.csr_matrix(X_test[hf].values)]).toarray())
for d in [X_train, X_val, X_test]: d.columns = [str(c) for c in d.columns]
num_cols = X_train.columns.tolist(); cat_cols = []
preprocessor = ColumnTransformer([('num', Pipeline([('imp',SimpleImputer(strategy='median')),('sc',StandardScaler())]), num_cols)], remainder='drop')
print(f'Features: {X_train.shape[1]} (500 TF-IDF + 2 handcrafted)')


## 18. Baseline Model


In [ ]:
results = {}
is_multi = y_train.nunique() > 2
avg = 'macro' if is_multi else 'binary'
for name, clf in [('Dummy', DummyClassifier(strategy='most_frequent', random_state=SEED)),
                  ('LogReg', LogisticRegression(max_iter=1000, random_state=SEED)),
                  ('RF', RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1))]:
    pipe = Pipeline([('pre',preprocessor),('clf',clf)])
    pipe.fit(X_train, y_train); yp = pipe.predict(X_val)
    r = {'Accuracy': accuracy_score(y_val,yp), 'F1': f1_score(y_val,yp,average=avg,zero_division=0)}
    results[name] = r; print(f'{name}: {r}')


## 19. LazyPredict Benchmark


In [ ]:
X_train_lp = preprocessor.fit_transform(X_train)
X_val_lp = preprocessor.transform(X_val)
lazy = LazyClassifier(verbose=0, ignore_warnings=True, random_state=SEED)
models_lp, _ = lazy.fit(X_train_lp, X_val_lp, y_train, y_val)
models_lp.head(15)


## 20. FLAML AutoML Run


In [ ]:
automl = AutoML()
automl.fit(X_train_lp, y_train, task='classification', metric='f1',
           time_budget=FLAML_BUDGET, seed=SEED, verbose=0)
print(f'Best: {automl.best_estimator}')
yp_fl = automl.predict(X_val_lp)
results['FLAML'] = {'Accuracy': accuracy_score(y_val, yp_fl), 'F1': f1_score(y_val, yp_fl, average=avg, zero_division=0)}
print(results['FLAML'])


## 21. Top 3 Model Selection


In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm = True
except ImportError: has_lgbm = False
try:
    from xgboost import XGBClassifier; has_xgb = True
except ImportError: has_xgb = False
top3 = {}
if has_lgbm: top3['LightGBM'] = LGBMClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1, n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3['ExtraTrees'] = ExtraTreesClassifier(n_estimators=400, random_state=SEED, n_jobs=-1)
if has_xgb: top3['XGBoost'] = XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0, n_jobs=-1, eval_metric='logloss')
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3['AdaBoost'] = AdaBoostClassifier(n_estimators=200, random_state=SEED)
top3['GradBoosting'] = GradientBoostingClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, random_state=SEED)
print(f'Top 3: {list(top3.keys())}')


## 22. Final Training and Evaluation of Top 3


In [ ]:
X_tr_t = preprocessor.fit_transform(X_train)
X_te_t = preprocessor.transform(X_test)
final = {}
for name, mdl in top3.items():
    mdl.fit(X_tr_t, y_train)
    yp = mdl.predict(X_te_t)
    final[name] = {'Accuracy': accuracy_score(y_test,yp),
                   'F1-macro': f1_score(y_test,yp,average='macro',zero_division=0),
                   'F1-weighted': f1_score(y_test,yp,average='weighted',zero_division=0)}
    print(f'\n{name}:'); print(classification_report(y_test,yp))
pd.DataFrame(final).T


In [ ]:
fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes, top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, m.predict(X_te_t), ax=ax, cmap='Blues')
    ax.set_title(n)
plt.tight_layout(); plt.show()


In [ ]:
if y_test.nunique() == 2:
    fig, ax = plt.subplots(figsize=(8,5))
    for n,m in top3.items():
        RocCurveDisplay.from_estimator(m, X_te_t, y_test, ax=ax, name=n)
    ax.plot([0,1],[0,1],'k--',alpha=0.5); ax.set_title('ROC Curves')
    plt.tight_layout(); plt.show()
else:
    print('Multi-class: ROC omitted for brevity.')


## 23. Error Analysis


In [ ]:
bm = list(top3.values())[0]; ypb = bm.predict(X_te_t)
print(f'Misclassified: {(y_test.values!=ypb).sum()}/{len(y_test)} ({(y_test.values!=ypb).mean():.1%})')
if y_test.nunique() == 2:
    fn = (y_test.values==1) & (ypb==0); fp = (y_test.values==0) & (ypb==1)
    print(f'FN (misclassified): {fn.sum()}')
    print(f'FP (wrong category): {fp.sum()}')


## 24. Interpretation and Business Insight

- **Topic-specific vocabulary** drives classification via TF-IDF
- **Sports/politics** are easiest categories (distinctive vocabulary)
- **Overlapping categories** (lifestyle/wellness) are hardest
- Headline length correlates weakly with category


## 25. Limitations

1. TF-IDF ignores word order
2. Top 8 categories only
3. English HuffPost only
4. Headlines may be ambiguous
5. No deep learning comparison


## 26. How to Improve This Project

1. Sentence transformers (SBERT)
2. All categories with hierarchical classification
3. Use article body text
4. Multi-label classification
5. Transfer learning from news-pretrained models


## 27. Production Considerations

- Real-time article tagging
- Vocabulary drift monitoring
- New category detection
- Multi-language support
- Editorial override workflow


## 28. Common Mistakes

1. Using accuracy for imbalanced multi-class
2. Not reducing category count
3. TF-IDF on all data (leakage)
4. Ignoring category overlap
5. Not using stratified split


## 29. Mini Challenge / Exercises

1. Try all categories instead of top 8
2. Headline-only vs headline+description
3. Use TruncatedSVD for dimensionality reduction
4. Confusion matrix analysis for similar categories
5. Train a Naive Bayes baseline


## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | Multi-class news classification |
| Dataset | ~200k articles |
| Difficulty | Medium |
| Key | TF-IDF captures category vocabulary |

Multi-class text classification with tabularized features.
